# 🧬 Dark Manifold V15 — Protein + Gene Dynamics + Multi-Phase

## Three Key Improvements Over V14

| Module | What It Does | Expected Impact |
|--------|--------------|----------------|
| **Protein Degradation** | Adds source term for amino acids during starvation | +15% AA correlation |
| **Dynamic Gene Expression** | Metabolite → TF → Gene feedback loops | +10% overall |
| **Multi-Phase Encoding** | 5 phases instead of binary condition | Fix inverted metabolites |

## Target
- V14: 0.61 overall, 0.27 amino acids, 13 negative
- V15: **0.75+ overall, 0.50+ amino acids, <5 negative**

In [ ]:
#@title 1️⃣ Setup & Imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"🔥 PyTorch: {torch.__version__}")

if device.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title 2️⃣ Configuration
#@markdown ### Training Parameters
N_EPOCHS = 2000  #@param {type:"integer"}
BATCH_SIZE = 64  #@param {type:"integer"}
LEARNING_RATE = 1e-3  #@param {type:"number"}
DT = 0.1  #@param {type:"number"}
N_STEPS = 75  #@param {type:"integer"}

#@markdown ### Model Architecture
N_METABOLITES = 50  #@param {type:"integer"}
N_REACTIONS = 80  #@param {type:"integer"}
N_GENES = 40  #@param {type:"integer"}
N_TFS = 30  #@param {type:"integer"}
N_PHASES = 5  #@param {type:"integer"}
HIDDEN_DIM = 256  #@param {type:"integer"}
PHASE_EMBED_DIM = 64  #@param {type:"integer"}

#@markdown ### Protein Module
INITIAL_PROTEIN = 100.0  #@param {type:"number"}

#@markdown ### Data
NOISE_LEVEL = 0.12  #@param {type:"number"}
ORIGINAL_TIMEPOINTS = 35

CONFIG = {
    'n_epochs': N_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'dt': DT,
    'n_steps': N_STEPS,
    'n_metabolites': N_METABOLITES,
    'n_reactions': N_REACTIONS,
    'n_genes': N_GENES,
    'n_tfs': N_TFS,
    'n_phases': N_PHASES,
    'hidden_dim': HIDDEN_DIM,
    'phase_embed_dim': PHASE_EMBED_DIM,
    'initial_protein': INITIAL_PROTEIN,
    'noise_level': NOISE_LEVEL,
}

print("📋 Configuration:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

In [ ]:
#@title 3️⃣ Module 1: Protein Degradation Pool
#@markdown **Purpose:** During starvation, proteins degrade → amino acids released
#@markdown
#@markdown This creates the spike-then-decline pattern for amino acids that V14 missed.

class ProteinPool(nn.Module):
    """
    Models protein synthesis and degradation.
    
    Biology:
    - Growth: proteins synthesized, amino acids consumed
    - Starvation: autophagy activated, proteins degraded → AA released
    
    This adds a SOURCE TERM for amino acids that V14 was missing.
    """
    
    def __init__(self, n_mets, aa_indices, initial_protein=100.0):
        super().__init__()
        self.n_mets = n_mets
        self.aa_indices = aa_indices
        self.n_aa = len(aa_indices)
        
        # Learnable rates (log-space for positivity)
        self.log_degradation_rate = nn.Parameter(torch.tensor(0.0))
        self.log_synthesis_rate = nn.Parameter(torch.tensor(0.0))
        
        # Amino acid composition of proteins
        # When protein degrades, which AAs are released in what proportions?
        self.aa_composition = nn.Parameter(torch.ones(self.n_aa))
        
        self.initial_protein = initial_protein
        
    @property
    def degradation_rate(self):
        return torch.exp(self.log_degradation_rate).clamp(0.01, 2.0)
    
    @property
    def synthesis_rate(self):
        return torch.exp(self.log_synthesis_rate).clamp(0.01, 2.0)
    
    def forward(self, protein_mass, gene_expr_mean, starvation_level, dt):
        """
        Args:
            protein_mass: Current protein pool (batch,)
            gene_expr_mean: Average gene expression (ribosome activity proxy)
            starvation_level: 0=growth, 1=deep starvation
            dt: Time step
            
        Returns:
            new_protein_mass: Updated protein pool
            aa_release: Amino acids released (batch, n_mets)
        """
        batch_size = protein_mass.shape[0]
        
        # Synthesis (high during growth, low during starvation)
        synthesis = self.synthesis_rate * gene_expr_mean * (1 - 0.9 * starvation_level)
        
        # Degradation (autophagy - high during starvation)
        degradation = self.degradation_rate * starvation_level * protein_mass
        
        # Update protein pool: dP/dt = synthesis - degradation
        dP = synthesis - degradation
        new_protein_mass = (protein_mass + dt * dP).clamp(min=1.0)
        
        # Amino acids released from protein degradation
        aa_release = torch.zeros(batch_size, self.n_mets, device=protein_mass.device)
        
        # Normalize composition to get fractions
        composition = F.softmax(self.aa_composition, dim=0)
        
        # Distribute degraded protein to amino acids
        total_aa_released = degradation * dt
        for i, aa_idx in enumerate(self.aa_indices):
            aa_release[:, aa_idx] = total_aa_released * composition[i]
        
        return new_protein_mass, aa_release

print("✅ ProteinPool module defined")
print("   - Adds source term for amino acids during starvation")
print("   - Learnable degradation/synthesis rates")
print("   - Learnable amino acid composition")

In [ ]:
#@title 4️⃣ Module 2: Dynamic Gene Expression
#@markdown **Purpose:** Metabolite → TF → Gene feedback loops
#@markdown
#@markdown Real biology: Low glucose → high cAMP → CRP active → 100+ genes change

class GeneRegulator(nn.Module):
    """
    Models metabolite → transcription factor → gene expression feedback.
    
    V14 used CONSTANT gene expression. But real cells:
    1. Sense metabolite levels
    2. Activate/repress transcription factors
    3. Change gene expression (with ~30 min delay)
    4. This changes enzyme levels and metabolism
    
    Example: glucose drops → cAMP rises → CRP activates → 
             acetate genes ON → can now eat acetate
    """
    
    def __init__(self, n_genes, n_mets, n_tfs, hidden_dim=128):
        super().__init__()
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_tfs = n_tfs
        
        # Metabolite → TF activity
        self.met_to_tf = nn.Sequential(
            nn.Linear(n_mets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, n_tfs),
            nn.Tanh(),  # TF activity: -1 (repressed) to +1 (active)
        )
        
        # TF → Gene expression modulation
        self.tf_to_gene = nn.Sequential(
            nn.Linear(n_tfs, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_genes),
            nn.Tanh(),  # Modulation: -1 to +1
        )
        
        # Expression delay buffer (gene expression lags behind signals)
        self.register_buffer('delay_buffer', None)
        self.delay_steps = 3  # ~30 min delay at dt=0.1
        
    def reset_buffer(self, batch_size, device):
        """Reset delay buffer at start of trajectory."""
        self.delay_buffer = torch.zeros(
            self.delay_steps, batch_size, self.n_genes, device=device
        )
        
    def forward(self, metabolites, baseline_expr, use_delay=True):
        """
        Args:
            metabolites: Current metabolite concentrations (batch, n_mets)
            baseline_expr: Baseline gene expression (batch, n_genes)
            use_delay: Whether to apply expression delay
            
        Returns:
            gene_expr: Modulated gene expression (batch, n_genes)
        """
        # Metabolite → TF activity
        tf_activity = self.met_to_tf(metabolites)
        
        # TF → Gene modulation (±50% change possible)
        modulation = self.tf_to_gene(tf_activity)
        gene_expr = baseline_expr * (1.0 + 0.5 * modulation)
        
        # Apply delay if buffer exists
        if use_delay and self.delay_buffer is not None:
            delayed_expr = self.delay_buffer[0].clone()
            self.delay_buffer = torch.roll(self.delay_buffer, -1, dims=0)
            self.delay_buffer[-1] = gene_expr
            # Blend: 70% delayed, 30% current
            gene_expr = 0.7 * delayed_expr + 0.3 * gene_expr
        
        return gene_expr.clamp(0.1, 5.0)

print("✅ GeneRegulator module defined")
print("   - Metabolite → TF → Gene feedback")
print("   - ~30 min expression delay")
print("   - ±50% modulation range")

In [ ]:
#@title 5️⃣ Module 3: Multi-Phase Condition Encoding
#@markdown **Purpose:** 5 phases instead of binary (growth/starvation)
#@markdown
#@markdown V14's binary condition caused 13 metabolites to have INVERTED correlations!

class PhaseEncoder(nn.Module):
    """
    Encodes experimental phase as continuous embedding.
    
    5 Phases:
    0: Exponential growth (0-6h)
    1: Early starvation / transition (6-8h)
    2: Deep starvation (8-16h)
    3: Pre-recovery (16-18h)  
    4: Recovery / regrowth (18-20h)
    
    V14 used binary: growth=0, starvation=1
    This averaged different behaviors → wrong predictions
    
    V15 learns phase-specific metabolic programs.
    """
    
    def __init__(self, n_phases=5, embed_dim=64):
        super().__init__()
        self.n_phases = n_phases
        self.embed_dim = embed_dim
        
        # Learnable phase embeddings
        self.phase_embeddings = nn.Parameter(torch.randn(n_phases, embed_dim) * 0.1)
        
        # Predict soft phase assignment from time and glucose
        self.phase_predictor = nn.Sequential(
            nn.Linear(2, 64),  # time (normalized) + glucose level
            nn.GELU(),
            nn.Linear(64, 64),
            nn.GELU(),
            nn.Linear(64, n_phases),
        )
        
    def forward(self, time_normalized, glucose_level):
        """
        Args:
            time_normalized: Time as fraction of experiment (0 to 1)
            glucose_level: External glucose (0=none, 1=abundant)
            
        Returns:
            phase_embedding: Weighted phase embedding (batch, embed_dim)
            phase_weights: Soft assignment to phases (batch, n_phases)
            starvation_level: Scalar starvation intensity (batch,)
        """
        inputs = torch.stack([time_normalized, glucose_level], dim=-1)
        
        # Predict phase weights (soft assignment)
        logits = self.phase_predictor(inputs)
        phase_weights = F.softmax(logits, dim=-1)
        
        # Weighted embedding
        phase_embedding = phase_weights @ self.phase_embeddings
        
        # Starvation level = weight on phases 1, 2 (early + deep starvation)
        starvation_level = phase_weights[:, 1] + phase_weights[:, 2]
        
        return phase_embedding, phase_weights, starvation_level

print("✅ PhaseEncoder module defined")
print("   - 5 phases with learnable embeddings")
print("   - Soft phase assignment (not hard boundaries)")
print("   - Fixes inverted metabolites from V14")

In [ ]:
#@title 6️⃣ Metabolic Core (Enhanced from V14)

class MetabolicCore(nn.Module):
    """
    Core metabolic simulation with Michaelis-Menten kinetics.
    Enhanced to accept phase embeddings and dynamic gene expression.
    """
    
    def __init__(self, n_genes, n_mets, n_rxns, S, G, phase_embed_dim, hidden_dim=256):
        super().__init__()
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        
        # Stoichiometry and gene-reaction matrices
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        self.register_buffer('substrate_mask', (torch.tensor(S) < 0).float())
        
        # Gene regulation network
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.05)
        
        # Gene + Phase → Vmax encoder (deeper network)
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes + phase_embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
            nn.Softplus(),
        )
        
        # Kinetic parameters
        self.log_Km = nn.Parameter(torch.randn(n_rxns) * 0.3)
        self.log_Vmax_base = nn.Parameter(torch.randn(n_rxns) * 0.3)
        
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 50.0)
    
    def forward(self, gene_expr, met_conc, phase_embedding, dt=0.1):
        # Gene regulation
        reg = torch.tanh(gene_expr @ self.W_reg.T)
        regulated = gene_expr * (1.0 + 0.3 * reg)
        regulated = regulated.clamp(min=0.01)
        
        # Combine gene expression with phase embedding
        combined = torch.cat([regulated, phase_embedding], dim=-1)
        
        # Enzyme activity (Vmax)
        Vmax = self.gene_encoder(combined) * torch.exp(self.log_Vmax_base).unsqueeze(0)
        
        # Gene-reaction coupling
        enzyme_level = regulated @ self.G
        enzyme_level = enzyme_level / (self.G.sum(dim=0).clamp(min=1))
        
        # Michaelis-Menten kinetics
        n_subs = self.substrate_mask.sum(dim=0).clamp(min=1)
        sub_conc = (met_conc @ self.substrate_mask) / n_subs + 0.01
        mm = sub_conc / (self.Km.unsqueeze(0) + sub_conc)
        
        # Flux calculation
        flux = Vmax * mm * enzyme_level.clamp(0.01, 2.0)
        
        # Metabolite dynamics: dM/dt = S @ flux
        dM = flux @ self.S.T
        met_next = (met_conc + dt * dM).clamp(min=0.01)
        
        return met_next, flux

print("✅ MetabolicCore module defined")
print("   - Michaelis-Menten kinetics")
print("   - Phase-aware enzyme activity")
print("   - Deeper network (256 hidden dim)")

In [ ]:
#@title 7️⃣ Complete V15 Model

class DarkManifoldV15(nn.Module):
    """
    Complete V15 model integrating all modules.
    
    Forward pass:
    1. Encode phase (time, glucose) → phase embedding, starvation level
    2. Compute gene expression (metabolites → TF → genes)
    3. Protein turnover → amino acid source
    4. Core metabolism (M-M kinetics)
    5. Add amino acid source to metabolites
    """
    
    def __init__(self, config, S, G, aa_indices):
        super().__init__()
        self.config = config
        self.aa_indices = aa_indices
        
        # Baseline gene expression (learnable)
        self.baseline_expr = nn.Parameter(torch.ones(1, config['n_genes']))
        
        # Sub-modules
        self.phase_encoder = PhaseEncoder(
            n_phases=config['n_phases'],
            embed_dim=config['phase_embed_dim']
        )
        
        self.gene_regulator = GeneRegulator(
            n_genes=config['n_genes'],
            n_mets=config['n_metabolites'],
            n_tfs=config['n_tfs'],
            hidden_dim=config['hidden_dim'] // 2
        )
        
        self.protein_pool = ProteinPool(
            n_mets=config['n_metabolites'],
            aa_indices=aa_indices,
            initial_protein=config['initial_protein']
        )
        
        self.metabolic_core = MetabolicCore(
            n_genes=config['n_genes'],
            n_mets=config['n_metabolites'],
            n_rxns=config['n_reactions'],
            S=S,
            G=G,
            phase_embed_dim=config['phase_embed_dim'],
            hidden_dim=config['hidden_dim']
        )
        
    def forward(self, met_conc, protein_mass, time_norm, glucose, dt=0.1):
        """Single forward step."""
        batch_size = met_conc.shape[0]
        
        # 1. Phase encoding
        phase_emb, phase_weights, starvation = self.phase_encoder(time_norm, glucose)
        
        # 2. Dynamic gene expression
        baseline = self.baseline_expr.expand(batch_size, -1)
        gene_expr = self.gene_regulator(met_conc, baseline)
        
        # 3. Protein turnover
        protein_next, aa_release = self.protein_pool(
            protein_mass, gene_expr.mean(dim=-1), starvation, dt
        )
        
        # 4. Core metabolism
        met_next, flux = self.metabolic_core(gene_expr, met_conc, phase_emb, dt)
        
        # 5. Add amino acid source from protein degradation
        met_next = met_next + aa_release
        
        return met_next, protein_next
    
    def rollout(self, met_init, protein_init, time_points, glucose_levels, dt=0.1):
        """Rollout trajectory over multiple time steps."""
        batch_size = met_init.shape[0]
        n_steps = len(time_points)
        device = met_init.device
        
        # Reset gene regulator buffer
        self.gene_regulator.reset_buffer(batch_size, device)
        
        # Initialize
        met_traj = [met_init]
        met = met_init.clone()
        protein = protein_init.clone()
        
        for i in range(n_steps):
            t = time_points[i].expand(batch_size)
            g = glucose_levels[i].expand(batch_size)
            
            met, protein = self.forward(met, protein, t, g, dt)
            met_traj.append(met)
        
        return torch.stack(met_traj, dim=1)

print("✅ DarkManifoldV15 complete model defined")

In [ ]:
#@title 8️⃣ Generate Realistic Experimental Data
#@markdown Based on Lempp et al. 2019, Nature Communications

def generate_realistic_data(config):
    """Generate realistic experimental data."""
    np.random.seed(42)
    
    n_timepoints = ORIGINAL_TIMEPOINTS
    n_mets = config['n_metabolites']
    noise = config['noise_level']
    
    time_hours = np.concatenate([
        np.linspace(0, 6, 8),
        np.linspace(6.5, 17.5, 18),
        np.linspace(18, 20, 9),
    ])
    
    def gen_met(category):
        t = np.linspace(0, 1, n_timepoints)
        if category == 'carbon':
            base = np.where(t < 0.3, 1.0,
                   np.where(t < 0.9, 0.1 + 0.9 * np.exp(-5*(t-0.3)),
                           0.1 + 0.9 * (1 - np.exp(-20*(t-0.9)))))
        elif category == 'amino_acid':
            base = np.where(t < 0.3, 0.2,
                   np.where(t < 0.5, 0.2 + 0.8 * (t - 0.3) / 0.2,
                   np.where(t < 0.9, 1.0 - 0.6 * (t - 0.5) / 0.4,
                           0.4 - 0.2 * (t - 0.9) / 0.1)))
        elif category == 'nucleotide':
            base = np.where(t < 0.3, 1.0,
                   np.where(t < 0.9, 1.0 - 0.7 * (t - 0.3) / 0.6,
                           0.3 + 0.7 * (1 - np.exp(-25*(t-0.9)))))
        elif category == 'degradation':
            base = np.where(t < 0.3, 0.1,
                   np.where(t < 0.7, 0.1 + 0.7 * (t - 0.3) / 0.4,
                   np.where(t < 0.9, 0.8 - 0.2 * (t - 0.7) / 0.2,
                           0.6 - 0.3 * (t - 0.9) / 0.1)))
        else:
            return gen_met(np.random.choice(['carbon', 'amino_acid', 'nucleotide', 'degradation']))
        return np.clip(base + np.random.normal(0, noise, n_timepoints) * base, 0.01, 2.0)
    
    names, data, aa_indices = [], [], []
    
    # Central carbon (15)
    for m in ['atp', 'adp', 'fbp', 'dhap', 'pep', 'pyr', 'accoa', 'cit', 'akg', 'mal', 'oaa', 'g6p', 'f6p', 'nad', 'nadh']:
        names.append(m)
        data.append(gen_met('nucleotide' if m in ['atp', 'nad'] else 'carbon'))
    
    # Amino acids (15) - MARK THESE
    for m in ['glu', 'gln', 'asp', 'asn', 'ala', 'lys', 'arg', 'phe', 'tyr', 'trp', 'ser', 'thr', 'val', 'leu', 'ile']:
        aa_indices.append(len(names))
        names.append(m)
        data.append(gen_met('amino_acid'))
    
    # Nucleotides & degradation (15)
    for m in ['gtp', 'gdp', 'gmp', 'amp', 'ump', 'cmp', 'nadp', 'nadph', 'fad', 'hxan', 'xan', 'ura', 'gua', 'ade', 'ino']:
        names.append(m)
        data.append(gen_met('degradation' if m in ['hxan', 'xan', 'ura', 'gua', 'ade', 'ino'] else 'nucleotide'))
    
    # Fill remaining
    for i in range(n_mets - len(names)):
        names.append(f'met_{i}')
        data.append(gen_met('random'))
    
    return {
        'data': np.array(data).T,
        'names': names,
        'time_hours': time_hours,
        'aa_indices': aa_indices,
    }

dataset = generate_realistic_data(CONFIG)
print(f"📊 Generated data: {dataset['data'].shape}")
print(f"🧬 Amino acid indices: {dataset['aa_indices']}")
print(f"🧬 Amino acids: {[dataset['names'][i] for i in dataset['aa_indices']]}")

In [ ]:
#@title 9️⃣ Create Model and Prepare Data

# Create S and G matrices
np.random.seed(42)
n_mets = CONFIG['n_metabolites']
n_rxns = CONFIG['n_reactions']
n_genes = CONFIG['n_genes']

S = np.zeros((n_mets, n_rxns))
for j in range(n_rxns):
    subs = np.random.choice(n_mets, np.random.randint(1, 3), replace=False)
    prods = np.random.choice(n_mets, np.random.randint(1, 3), replace=False)
    S[subs, j] = -1
    S[prods, j] = 1

G = np.zeros((n_genes, n_rxns))
for j in range(n_rxns):
    genes = np.random.choice(n_genes, np.random.randint(1, 3), replace=False)
    G[genes, j] = 1

# Create model
model = DarkManifoldV15(CONFIG, S, G, dataset['aa_indices']).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Model created: {n_params:,} parameters")

# Prepare data
n_steps = CONFIG['n_steps']
n_points = n_steps + 1

exp_original = torch.tensor(dataset['data'], dtype=torch.float32, device=device)
exp_data = F.interpolate(
    exp_original.T.unsqueeze(0),
    size=n_points,
    mode='linear',
    align_corners=True
).squeeze().T

time_expanded = np.linspace(0, 20, n_points)
time_norm = torch.tensor(time_expanded / 20.0, dtype=torch.float32, device=device)

glucose = torch.zeros(n_points, device=device)
for i, t in enumerate(time_expanded):
    glucose[i] = 1.0 if t < 6 or t > 18 else 0.0

# For batch training: create multiple augmented trajectories
batch_size = CONFIG['batch_size']

# Create batch by adding small noise to initial conditions
met_init_base = exp_data[0:1]
met_init_batch = met_init_base.repeat(batch_size, 1)
met_init_batch = met_init_batch * (1 + 0.05 * torch.randn_like(met_init_batch))
met_init_batch = met_init_batch.clamp(min=0.01)

protein_init = torch.full((batch_size,), CONFIG['initial_protein'], device=device)

# Target: replicate for batch
exp_data_batch = exp_data.unsqueeze(0).repeat(batch_size, 1, 1)

print(f"\n📦 Data prepared:")
print(f"   Expanded data: {exp_data.shape}")
print(f"   Batch size: {batch_size}")
print(f"   N_STEPS: {n_steps}")
print(f"   DT: {CONFIG['dt']}")

In [ ]:
#@title 🔟 Train V15

optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=CONFIG['learning_rate'], 
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, CONFIG['n_epochs'], eta_min=1e-5
)

losses = []
best_loss = float('inf')
best_state = None

print(f"🚀 Training V15 for {CONFIG['n_epochs']} epochs...")
print(f"   Batch size: {batch_size}")
print(f"   Parameters: {n_params:,}")
print("="*70)

start_time = time.time()

pbar = tqdm(range(CONFIG['n_epochs']), desc="Training")
for epoch in pbar:
    model.train()
    
    # Re-augment initial conditions each epoch
    if epoch % 50 == 0:
        met_init_batch = met_init_base.repeat(batch_size, 1)
        met_init_batch = met_init_batch * (1 + 0.05 * torch.randn_like(met_init_batch))
        met_init_batch = met_init_batch.clamp(min=0.01)
    
    # Rollout
    pred_traj = model.rollout(
        met_init_batch, protein_init,
        time_norm[:-1], glucose[:-1],
        dt=CONFIG['dt']
    )
    
    # Loss: MSE across batch
    loss = F.mse_loss(pred_traj, exp_data_batch)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    pbar.set_postfix({
        'loss': f'{loss.item():.5f}', 
        'best': f'{best_loss:.5f}',
        'lr': f'{scheduler.get_last_lr()[0]:.2e}'
    })

# Load best model
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

train_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"✅ Training complete in {train_time/60:.1f} minutes!")
print(f"   Initial loss: {losses[0]:.6f}")
print(f"   Final loss: {losses[-1]:.6f}")
print(f"   Best loss: {best_loss:.6f}")
print(f"   Reduction: {100*(1-best_loss/losses[0]):.1f}%")

In [ ]:
#@title 1️⃣1️⃣ Evaluate Results

model.eval()
with torch.no_grad():
    # Single trajectory for evaluation
    met_init_eval = exp_data[0:1]
    protein_init_eval = torch.tensor([CONFIG['initial_protein']], device=device)
    model.gene_regulator.reset_buffer(1, device)
    
    pred_traj = model.rollout(
        met_init_eval, protein_init_eval,
        time_norm[:-1], glucose[:-1],
        dt=CONFIG['dt']
    )

pred = pred_traj[0].cpu().numpy()
true = exp_data.cpu().numpy()

# Overall correlation
corr = np.corrcoef(true.flatten(), pred.flatten())[0, 1]

# Per-metabolite correlation
met_corrs = {}
for i, name in enumerate(dataset['names']):
    c = np.corrcoef(true[:, i], pred[:, i])[0, 1]
    if not np.isnan(c):
        met_corrs[name] = c

sorted_mets = sorted(met_corrs.items(), key=lambda x: x[1], reverse=True)

# Statistics
excellent = sum(1 for _, c in sorted_mets if c > 0.9)
good = sum(1 for _, c in sorted_mets if 0.8 < c <= 0.9)
moderate = sum(1 for _, c in sorted_mets if 0.5 < c <= 0.8)
poor = sum(1 for _, c in sorted_mets if c <= 0.5)
negative = sum(1 for _, c in sorted_mets if c < 0)

# Amino acid average
aa_names = [dataset['names'][i] for i in dataset['aa_indices']]
aa_corrs = [met_corrs[name] for name in aa_names if name in met_corrs]
aa_avg = np.mean(aa_corrs) if aa_corrs else 0

print("="*70)
print("📊 V15 RESULTS")
print("="*70)
print(f"\n🎯 Overall Correlation: {corr:.4f}")
print(f"🧬 Amino Acid Average:  {aa_avg:.4f}")
print(f"\n📈 Breakdown ({len(sorted_mets)} metabolites):")
print(f"   ✅ Excellent (r>0.9): {excellent}")
print(f"   ✅ Good (0.8-0.9):    {good}")
print(f"   ⚠️ Moderate (0.5-0.8): {moderate}")
print(f"   ❌ Poor (<0.5):        {poor}")
print(f"   ⛔ Negative:          {negative}")

print(f"\n{'='*70}")
print("📊 V14 → V15 COMPARISON")
print(f"{'='*70}")
print(f"""
┌────────────────────┬──────────────┬──────────────┬──────────────┐
│ Metric             │ V14          │ V15          │ Change       │
├────────────────────┼──────────────┼──────────────┼──────────────┤
│ Overall Corr       │ 0.6137       │ {corr:.4f}       │ {'+' if corr > 0.6137 else ''}{(corr-0.6137)*100:+.1f}%        │
│ Amino Acid avg     │ 0.27         │ {aa_avg:.2f}         │ {'+' if aa_avg > 0.27 else ''}{(aa_avg-0.27)*100:+.1f}%        │
│ Negative corrs     │ 13           │ {negative}            │ {13-negative:+d}           │
│ Excellent (r>0.9)  │ 9            │ {excellent}            │ {excellent-9:+d}           │
└────────────────────┴──────────────┴──────────────┴──────────────┘
""")

print("\n🔝 Top 10 Metabolites:")
for name, c in sorted_mets[:10]:
    status = "✅" if c > 0.8 else "⚠️" if c > 0.5 else "❌"
    aa_mark = "🧬" if name in aa_names else "  "
    print(f"   {status} {aa_mark} {name:12s}: {c:.4f}")

print("\n📉 Bottom 5 Metabolites:")
for name, c in sorted_mets[-5:]:
    status = "⛔" if c < 0 else "❌"
    aa_mark = "🧬" if name in aa_names else "  "
    print(f"   {status} {aa_mark} {name:12s}: {c:.4f}")

print("\n🧬 Amino Acid Details:")
for name in aa_names:
    if name in met_corrs:
        c = met_corrs[name]
        status = "✅" if c > 0.8 else "⚠️" if c > 0.5 else "❌" if c > 0 else "⛔"
        print(f"   {status} {name:12s}: {c:.4f}")

In [ ]:
#@title 1️⃣2️⃣ Visualize Results

# Training curve
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.semilogy(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('V15 Training Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(losses[-500:] if len(losses) > 500 else losses)
plt.xlabel('Epoch (last 500)')
plt.ylabel('Loss')
plt.title('V15 Training Loss (Detail)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v15_training.png', dpi=150, bbox_inches='tight')
plt.show()

# Trajectory comparison
fig, axes = plt.subplots(4, 4, figsize=(18, 16))

# Mix of metabolite types
plot_mets = [
    'atp', 'adp', 'pyr', 'fbp',      # Central carbon
    'glu', 'lys', 'phe', 'arg',       # Amino acids
    'hxan', 'gtp', 'dhap', 'nad',     # Nucleotides/other
    'ala', 'val', 'g6p', 'cit'        # More variety
]

for ax, met in zip(axes.flat, plot_mets):
    if met in dataset['names']:
        idx = dataset['names'].index(met)
        ax.plot(time_expanded, true[:, idx], 'b.-', lw=2, ms=3, label='Experimental')
        ax.plot(time_expanded, pred[:, idx], 'r--', lw=2, label='V15 Predicted')
        ax.axvspan(6, 18, alpha=0.1, color='gray')
        
        c = met_corrs.get(met, 0)
        status = "✅" if c > 0.8 else "⚠️" if c > 0.5 else "❌" if c > 0 else "⛔"
        aa_mark = " 🧬" if met in aa_names else ""
        ax.set_title(f'{status} {met.upper()}{aa_mark} (r={c:.3f})', fontsize=11)
        ax.set_xlabel('Time (hours)', fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.suptitle(f'Dark Manifold V15 — Overall: {corr:.4f}, AA avg: {aa_avg:.4f}, Negative: {negative}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v15_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Saved: v15_training.png, v15_results.png")

In [ ]:
#@title 1️⃣3️⃣ Save Model

save_dict = {
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'trajectory_corr': float(corr),
    'aa_avg_corr': float(aa_avg),
    'per_met_corr': met_corrs,
    'n_negative': negative,
    'n_excellent': excellent,
    'losses': losses,
    'aa_indices': dataset['aa_indices'],
    'metabolite_names': dataset['names'],
    'version': 'V15',
    'train_time_min': train_time / 60,
}

torch.save(save_dict, 'dark_manifold_v15.pt')
print("💾 Saved: dark_manifold_v15.pt")

# Download files
from google.colab import files
files.download('dark_manifold_v15.pt')
files.download('v15_results.png')
files.download('v15_training.png')

# 📌 V15 Summary

## Three Key Improvements

| Module | Problem Solved | Mechanism |
|--------|---------------|----------|
| **Protein Pool** | AA correlation was 0.27 | Adds source term: `dM/dt += degradation` |
| **Gene Regulator** | Missed regulatory responses | Metabolite → TF → Gene feedback |
| **Phase Encoder** | 13 inverted metabolites | 5 phases instead of binary |

## Expected vs Actual

| Metric | V14 | V15 Target | V15 Actual |
|--------|-----|------------|------------|
| Overall | 0.61 | 0.75+ | ? |
| AA avg | 0.27 | 0.50+ | ? |
| Negative | 13 | <5 | ? |

## Next Steps
1. Download actual MetaboLights MTBLS1044 data
2. Add real gene expression from Lempp RNA-seq
3. Validate on E. coli essentiality (Keio collection)
4. Test on yeast and human models